# 05 — NHiTS quantile neural model on Track A
Goal: capture year-over-year dynamics that the per-rm linear can't. Trained with `MQLoss(quantiles=[0.1, 0.2, 0.5])` on daily deliveries; cumsum at inference.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))
import warnings, logging; warnings.filterwarnings('ignore'); logging.getLogger('pytorch_lightning').setLevel(logging.WARNING)
import pandas as pd
from src.data import load_or_build, build_profile
from src.gating import assign_tracks
from src.models.nhits_quantile import NHITSQuantileForecaster
from src.models.linear_per_rm import PerRMLinearForecaster
from src.ensemble import blend, EnsembleConfig
from src.validation import DEFAULT_FOLDS, evaluate, build_query_for_fold


In [ ]:
ds = load_or_build()
rm_ids = sorted(ds.daily['rm_id'].unique().tolist())
for fold in DEFAULT_FOLDS:
    cutoff = fold.train_end + pd.Timedelta(days=1)
    daily_pre = ds.daily[ds.daily['date'] < cutoff]
    profile = build_profile(daily_pre, year=fold.target_year - 1)
    tracks = assign_tracks(profile, all_rm_ids=rm_ids)
    track_a = tracks[tracks['track']=='A']['rm_id'].tolist()
    print(f'{fold.name}: training NHITS on {len(track_a)} Track-A rm_ids')
    nh = NHITSQuantileForecaster(rm_ids_to_train=track_a, horizon=151, input_size=730, max_epochs=30, hidden_size=128, batch_size=64).fit(ds.daily, history_end=cutoff)
    end_dates = pd.date_range(f'{fold.target_year}-01-02', f'{fold.target_year}-05-31', freq='D')
    preds_nh = nh.predict_cumulative(target_year=fold.target_year, end_dates=end_dates)
    lin = PerRMLinearForecaster(fit_year=fold.target_year - 1, slope_shrink=0.6).fit(daily_pre)
    preds_lin = lin.predict(build_query_for_fold(fold, rm_ids))
    for w_lin in [1.0, 0.7, 0.5, 0.3, 0.0]:
        cfg = EnsembleConfig(track_weights={'A': {'linear': w_lin, 'nhits': 1-w_lin}, 'B': {'linear': 1.0}, 'C': {}, 'D': {}}, cap_multiplier=None)
        b = blend({'linear': preds_lin, 'nhits': preds_nh}, tracks, cfg)
        print(f'  Track A: linear={w_lin:.1f}/nhits={1-w_lin:.1f}  pinball={evaluate(b, fold, ds.daily)["mean_pinball"]:.0f}')
